# MedHallu Hallucination Risk Modeling


## 0. 설치

In [3]:
!pip install -q datasets sentence-transformers xgboost scikit-learn pandas numpy

## 1. 라이브러리 불러오기

In [4]:
import os
import re
import numpy as np
import pandas as pd

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    fbeta_score,
    brier_score_loss,
    average_precision_score
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier


## 2. MedHallu 데이터 로드

`pqa_artificial`은 train, `pqa_labeled`는 calibration/test

In [5]:
# 1. 학습/train용 load
ds_artificial = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_artificial")

# 2. 평가/test용 load
ds_labeled = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_labeled")

artificial_raw = ds_artificial["train"].to_pandas()
labeled_raw = ds_labeled["train"].to_pandas()

print("artificial_raw:", artificial_raw.shape)
print("labeled_raw:", labeled_raw.shape)
print("columns:", artificial_raw.columns.tolist())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/10.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

artificial_raw: (9000, 6)
labeled_raw: (1000, 6)
columns: ['Question', 'Knowledge', 'Ground Truth', 'Difficulty Level', 'Hallucinated Answer', 'Category of Hallucination']


## 3. 이진 분류 데이터 변환

- `Ground Truth` → factual answer, label 0
- `Hallucinated Answer` → hallucination answer, label 1

`Ground Truth`는 label 생성/결과 확인용으로만 사용하고, feature에는 사용X

In [6]:
def build_binary_medhal(df):
    """
    MedHallu 원자료를 binary classification 형태로 변환.

    Ground Truth -> label 0, Factual
    Hallucinated Answer -> label 1, Hallucination

    Ground Truth는 label 생성과 평가/결과 확인용으로만 사용한다.
    실제 모델 feature에서는 Ground Truth를 사용하지 않는다.
    """
    factual = df.copy()
    factual["Answer"] = factual["Ground Truth"]
    factual["label"] = 0

    hallucinated = df.copy()
    hallucinated["Answer"] = hallucinated["Hallucinated Answer"]
    hallucinated["label"] = 1

    out = pd.concat([factual, hallucinated], ignore_index=True)

    keep_cols = [
        "Question",
        "Knowledge",
        "Ground Truth",  # 결과 확인용으로만 유지
        "Answer",
        "Difficulty Level",
        "Category of Hallucination",
        "label"
    ]

    keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[keep_cols].dropna().reset_index(drop=True)
    return out

train_df_full = build_binary_medhal(artificial_raw)
labeled_df = build_binary_medhal(labeled_raw)

print("train_df_full:", train_df_full.shape)
print("labeled_df:", labeled_df.shape)
print("\ntrain label distribution")
print(train_df_full["label"].value_counts())
print("\nlabeled label distribution")
print(labeled_df["label"].value_counts())


train_df_full: (18000, 7)
labeled_df: (2000, 7)

train label distribution
label
0    9000
1    9000
Name: count, dtype: int64

labeled label distribution
label
0    1000
1    1000
Name: count, dtype: int64


## 4. Train / Calibration / Test 구성


In [7]:
USE_TRAIN_SAMPLE = False
N_PER_CLASS = 10000

if USE_TRAIN_SAMPLE:
    train_df = (
        train_df_full
        .groupby("label", group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), N_PER_CLASS), random_state=42))
        .reset_index(drop=True)
    )
else:
    train_df = train_df_full.copy()

# labeled data를 calibration / final test로 분리
calib_df, test_df = train_test_split(
    labeled_df,
    test_size=0.5,
    stratify=labeled_df["label"],
    random_state=42
)

print("train_df:", train_df.shape)
print("calib_df:", calib_df.shape)
print("test_df:", test_df.shape)

print("\ntrain label")
print(train_df["label"].value_counts())
print("\ncalib label")
print(calib_df["label"].value_counts())
print("\ntest label")
print(test_df["label"].value_counts())


train_df: (18000, 7)
calib_df: (1000, 7)
test_df: (1000, 7)

train label
label
0    9000
1    9000
Name: count, dtype: int64

calib label
label
1    500
0    500
Name: count, dtype: int64

test label
label
1    500
0    500
Name: count, dtype: int64


## 5. SBERT 모델 로드

In [8]:
sbert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 6. 공통 유틸 함수

Question, Knowledge, Answer를 각각 독립적으로 벡터화

| 함수                  | 역할                                         |
| ------------------- | ------------------------------------------ |
| `rowwise_cosine()`  | Question, Knowledge, Answer 벡터 간 의미 유사도 계산 |
| `split_sentences()` | Knowledge 텍스트를 문장 단위로 분리                   |
| 최종 목적               | 답변이 근거 지식과 얼마나 일치하는지 수치화                   |


In [9]:
def rowwise_cosine(a, b):
    numerator = np.sum(a * b, axis=1)
    denominator = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
    return numerator / np.clip(denominator, 1e-8, None)


def split_sentences(text):
    """간단한 영어 문장 분리 함수."""
    sents = re.split(r'(?<=[.!?])\s+', str(text))
    return [s.strip() for s in sents if len(s.strip()) > 0]


## 7. SBERT 관계 피처 생성
Knowledge와 Answer, Question과 Answer의 의미적 관계 측정



In [10]:
def sbert_features(df, batch_size=128):
    q_emb = sbert_model.encode(
        df["Question"].astype(str).tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    k_emb = sbert_model.encode(
        df["Knowledge"].astype(str).tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    a_emb = sbert_model.encode(
        df["Answer"].astype(str).tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    scalar_feat = pd.DataFrame({
        "sbert_cos_KA": rowwise_cosine(k_emb, a_emb),
        "sbert_cos_QA": rowwise_cosine(q_emb, a_emb),
        "diff_KA_mean": np.mean(np.abs(k_emb - a_emb), axis=1),
        "diff_KA_max": np.max(np.abs(k_emb - a_emb), axis=1),
        "diff_KA_l2": np.linalg.norm(k_emb - a_emb, axis=1),
    })

    diff_KA = np.abs(k_emb - a_emb)
    diff_KA_df = pd.DataFrame(
        diff_KA,
        columns=[f"diff_KA_vec_{i}" for i in range(diff_KA.shape[1])]
    )

    return pd.concat([scalar_feat, diff_KA_df], axis=1)


## 8. Knowledge top-k 문장 similarity 피처

Answer와 가장 유사한 Knowledge 문장을 찾아 top-k similarity를 피처로 사용

| Output 변수                 | 의미                                     |
| ------------------------- | -------------------------------------- |
| `top1_KA_sim`             | Answer와 가장 유사한 Knowledge 문장의 유사도       |
| `top3_KA_mean_sim`        | Answer와 유사한 상위 3개 Knowledge 문장 유사도의 평균 |
| `top3_KA_max_sim`         | 상위 3개 중 가장 높은 유사도                      |
| `top3_KA_min_sim`         | 상위 3개 중 가장 낮은 유사도                      |
| `num_knowledge_sentences` | Knowledge 안에 포함된 문장 수                  |


In [11]:
def topk_knowledge_sentence_features(df, top_k=3, batch_size=128):
    rows = []

    for _, row in df.iterrows():
        knowledge = str(row["Knowledge"])
        answer = str(row["Answer"])
        sentences = split_sentences(knowledge)

        if len(sentences) == 0:
            rows.append({
                "top1_KA_sim": 0.0,
                "top3_KA_mean_sim": 0.0,
                "top3_KA_max_sim": 0.0,
                "top3_KA_min_sim": 0.0,
                "num_knowledge_sentences": 0
            })
            continue

        sent_emb = sbert_model.encode(
            sentences,
            batch_size=batch_size,
            convert_to_numpy=True,
            show_progress_bar=False
        )
        ans_emb = sbert_model.encode(
            [answer],
            convert_to_numpy=True,
            show_progress_bar=False
        )

        ans_rep = np.repeat(ans_emb, len(sentences), axis=0)
        sims = rowwise_cosine(sent_emb, ans_rep)

        k = min(top_k, len(sims))
        top_sims = np.sort(sims)[-k:]

        rows.append({
            "top1_KA_sim": float(np.max(sims)),
            "top3_KA_mean_sim": float(np.mean(top_sims)),
            "top3_KA_max_sim": float(np.max(top_sims)),
            "top3_KA_min_sim": float(np.min(top_sims)),
            "num_knowledge_sentences": len(sentences)
        })

    return pd.DataFrame(rows)


## 9. 수치·단위·방향성·부정 불일치 피처

- 숫자 불일치(답변이 근거와 다른 숫자를 제시하는지)
- 단위 불일치(제약·바이오 문맥의 단위 오류)
- 증가/감소 방향성 불일치(increase/decrease, higher/lower 등 방향 반전 오류)
- 부정 표현 불일치(not, no, without 등 부정 표현 차이)

In [12]:
number_pattern = r"[-+]?\d*\.\d+|\d+"
unit_pattern = r"\b(mg|g|kg|ml|l|%|percent|dose|doses|μg|ug|mol|mmol|hours|days|weeks|months)\b"

increase_words = [
    "increase", "increased", "higher", "rise", "elevated",
    "upregulated", "greater", "improve", "improved", "enhanced"
]

decrease_words = [
    "decrease", "decreased", "lower", "reduced", "downregulated",
    "less", "decline", "declined", "worse", "worsened"
]

negation_words = ["not", "no", "none", "never", "without", "lack", "lacks", "failed", "failure", "absence"]


def extract_numbers(text):
    nums = re.findall(number_pattern, str(text))
    return [float(x) for x in nums]


def extract_units(text):
    return re.findall(unit_pattern, str(text).lower())


def direction_flag(text):
    text = str(text).lower()
    inc = any(w in text for w in increase_words)
    dec = any(w in text for w in decrease_words)

    if inc and not dec:
        return 1
    elif dec and not inc:
        return -1
    else:
        return 0


def negation_count(text):
    text = str(text).lower()
    tokens = re.findall(r"\b\w+\b", text)
    return sum(1 for t in tokens if t in negation_words)


def symbolic_mismatch_features(df):
    rows = []

    for _, row in df.iterrows():
        knowledge = str(row["Knowledge"])
        answer = str(row["Answer"])

        k_nums = extract_numbers(knowledge)
        a_nums = extract_numbers(answer)

        k_units = extract_units(knowledge)
        a_units = extract_units(answer)

        num_count_knowledge = len(k_nums)
        num_count_answer = len(a_nums)
        num_count_diff_KA = abs(num_count_knowledge - num_count_answer)

        if len(a_nums) > 0:
            mismatch_count_KA = sum([1 for x in a_nums if x not in k_nums])
            mismatch_ratio_KA = mismatch_count_KA / len(a_nums)
        else:
            mismatch_count_KA = 0
            mismatch_ratio_KA = 0

        unit_mismatch = int(set(a_units) - set(k_units) != set()) if len(a_units) > 0 else 0

        dir_k = direction_flag(knowledge)
        dir_a = direction_flag(answer)
        direction_mismatch_KA = int(dir_k != 0 and dir_a != 0 and dir_k != dir_a)

        neg_k = negation_count(knowledge)
        neg_a = negation_count(answer)
        negation_count_diff = abs(neg_k - neg_a)
        negation_mismatch = int((neg_k > 0) != (neg_a > 0))

        rows.append({
            "num_count_knowledge": num_count_knowledge,
            "num_count_answer": num_count_answer,
            "num_count_diff_KA": num_count_diff_KA,
            "num_mismatch_count_KA": mismatch_count_KA,
            "num_mismatch_ratio_KA": mismatch_ratio_KA,
            "unit_mismatch": unit_mismatch,
            "direction_mismatch_KA": direction_mismatch_KA,
            "negation_count_knowledge": neg_k,
            "negation_count_answer": neg_a,
            "negation_count_diff": negation_count_diff,
            "negation_mismatch": negation_mismatch,
        })

    return pd.DataFrame(rows)


## 10. 메타 피처

문장 길이와 답변/근거 길이 비율을 사용

`Difficulty Level`, `Category of Hallucination`은 실제 운영에서 사전에 알 수 없을 수 있으므로 기본값은 제외함

In [13]:
USE_CATEGORY_FEATURES = False


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def meta_features(df, encoder=None, fit=False, use_category_features=False):
    temp = df.copy()

    temp["question_len"] = temp["Question"].astype(str).str.len()
    temp["knowledge_len"] = temp["Knowledge"].astype(str).str.len()
    temp["answer_len"] = temp["Answer"].astype(str).str.len()
    temp["answer_knowledge_len_ratio"] = temp["answer_len"] / temp["knowledge_len"].clip(lower=1)

    numeric_meta = temp[
        [
            "question_len",
            "knowledge_len",
            "answer_len",
            "answer_knowledge_len_ratio"
        ]
    ].reset_index(drop=True)

    if not use_category_features:
        return numeric_meta, encoder

    cat_cols = []
    if "Difficulty Level" in temp.columns:
        cat_cols.append("Difficulty Level")
    if "Category of Hallucination" in temp.columns:
        cat_cols.append("Category of Hallucination")

    if len(cat_cols) > 0:
        cat_data = temp[cat_cols].astype(str)
        if fit:
            encoder = make_ohe()
            cat_encoded = encoder.fit_transform(cat_data)
        else:
            cat_encoded = encoder.transform(cat_data)

        cat_encoded_df = pd.DataFrame(
            cat_encoded,
            columns=encoder.get_feature_names_out(cat_cols)
        )
        return pd.concat([numeric_meta, cat_encoded_df], axis=1), encoder

    return numeric_meta, encoder


## 11. 전체 feature matrix 생성

모든 피처를 결합하여 모델 입력 데이터 생성

In [14]:
def build_features(df, encoder=None, fit_meta=False, use_category_features=False):
    print("1) SBERT relationship features 생성 중...")
    sb_feat = sbert_features(df)

    print("2) top-k Knowledge sentence features 생성 중...")
    topk_feat = topk_knowledge_sentence_features(df, top_k=3)

    print("3) symbolic mismatch features 생성 중...")
    sym_feat = symbolic_mismatch_features(df)

    print("4) meta features 생성 중...")
    meta_feat, encoder = meta_features(
        df,
        encoder=encoder,
        fit=fit_meta,
        use_category_features=use_category_features
    )

    X = pd.concat(
        [
            sb_feat.reset_index(drop=True),
            topk_feat.reset_index(drop=True),
            sym_feat.reset_index(drop=True),
            meta_feat.reset_index(drop=True)
        ],
        axis=1
    )

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    return X, encoder


## 12. Feature 생성

In [15]:
print("Generating features for training data...")
X_train, meta_encoder = build_features(
    train_df,
    fit_meta=True,
    use_category_features=USE_CATEGORY_FEATURES
)
y_train = train_df["label"].values
print("X_train shape:", X_train.shape)

Generating features for training data...
1) SBERT relationship features 생성 중...


Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

2) top-k Knowledge sentence features 생성 중...
3) symbolic mismatch features 생성 중...
4) meta features 생성 중...
X_train shape: (18000, 409)


In [19]:
X_calib, _ = build_features(
    calib_df,
    encoder=meta_encoder,
    fit_meta=False,
    use_category_features=USE_CATEGORY_FEATURES
)
y_calib = calib_df["label"].values
print("X_calib:", X_calib.shape)

1) SBERT relationship features 생성 중...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

2) top-k Knowledge sentence features 생성 중...
3) symbolic mismatch features 생성 중...
4) meta features 생성 중...
X_calib: (1000, 409)


In [16]:
print("Generating features for test data...")
X_test, _ = build_features(
    test_df,
    encoder=meta_encoder,
    fit_meta=False,
    use_category_features=USE_CATEGORY_FEATURES
)
y_test = test_df["label"].values
print("X_test shape:", X_test.shape)

Generating features for test data...
1) SBERT relationship features 생성 중...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

2) top-k Knowledge sentence features 생성 중...
3) symbolic mismatch features 생성 중...
4) meta features 생성 중...
X_test shape: (1000, 409)


## 13. XGBoost 학습

In [17]:
xgb_model = XGBClassifier(
    n_estimators=800,
    max_depth=3,
    learning_rate=0.02,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.1,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.02, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=800, n_jobs=-1,
              num_parallel_tree=None, ...)

## 14. Probability Calibration

`Hallucination Probability`보험 likelihood로 해석 가능한 확률값으로 보정

In [20]:
try:
    calibrated_model = CalibratedClassifierCV(
        estimator=xgb_model,
        method="sigmoid",
        cv="prefit"
    )
except TypeError:
    calibrated_model = CalibratedClassifierCV(
        base_estimator=xgb_model,
        method="sigmoid",
        cv="prefit"
    )

calibrated_model.fit(X_calib, y_calib)


/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


CalibratedClassifierCV(cv='prefit',
                       estimator=XGBClassifier(base_score=None, booster=None,
                                               callbacks=None,
                                               colsample_bylevel=None,
                                               colsample_bynode=None,
                                               colsample_bytree=0.85,
                                               device=None,
                                               early_stopping_rounds=None,
                                               enable_categorical=False,
                                               eval_metric='logloss',
                                               feature_types=None,
                                               feature_weights=None, gamma=0.1,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=0.02, max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None, max_depth=3,
                                               max_leaves=None,
                                               min_child_weight=3, missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=800, n_jobs=-1,
                                               num_parallel_tree=None, ...))

## 15. FN 중심 threshold 최적화

실제 환각 답변을 factual로 통과시키는 오류를 최소화

In [21]:
def find_best_threshold(y_true, y_prob, target_recall=0.85, beta=2):
    thresholds = np.linspace(0.01, 0.99, 99)
    results = []

    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)

        rec = recall_score(y_true, y_pred, zero_division=0)
        prec = precision_score(y_true, y_pred, zero_division=0)
        f2 = fbeta_score(y_true, y_pred, beta=beta, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fn_risk = fn / (fn + tp) if (fn + tp) > 0 else 0

        results.append({
            "threshold": th,
            "precision": prec,
            "recall": rec,
            "f2_score": f2,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
            "false_negative_risk": fn_risk
        })

    result_df = pd.DataFrame(results)
    candidates = result_df[result_df["recall"] >= target_recall]

    if len(candidates) > 0:
        best = candidates.sort_values("f2_score", ascending=False).iloc[0]
    else:
        best = result_df.sort_values("f2_score", ascending=False).iloc[0]

    return best, result_df

calib_prob = calibrated_model.predict_proba(X_calib)[:, 1]

best_threshold, threshold_table = find_best_threshold(
    y_calib,
    calib_prob,
    target_recall=0.85,
    beta=2
)

print(best_threshold)


threshold                0.150000
precision                0.596833
recall                   0.980000
f2_score                 0.868486
tn                     169.000000
fp                     331.000000
fn                      10.000000
tp                     490.000000
false_negative_risk      0.020000
Name: 14, dtype: float64


## 16. 최종 test 평가

모델의 hallucination 탐지력과 FN risk 평가

In [22]:
test_prob = calibrated_model.predict_proba(X_test)[:, 1]
final_threshold = best_threshold["threshold"]

test_pred = (test_prob >= final_threshold).astype(int)

acc = accuracy_score(y_test, test_pred)
auc = roc_auc_score(y_test, test_prob)
ap = average_precision_score(y_test, test_prob)
brier = brier_score_loss(y_test, test_prob)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
hallucination_false_negative_risk = fn / (fn + tp) if (fn + tp) > 0 else 0

print("Final threshold:", final_threshold)
print("Accuracy:", acc)
print("ROC-AUC:", auc)
print("PR-AUC / Average Precision:", ap)
print("Brier Score:", brier)
print("Confusion Matrix")
print(confusion_matrix(y_test, test_pred))
print("Hallucination False Negative Risk:", hallucination_false_negative_risk)
print()
print(classification_report(y_test, test_pred))


Final threshold: 0.15000000000000002
Accuracy: 0.616
ROC-AUC: 0.830796
PR-AUC / Average Precision: 0.826365779465685
Brier Score: 0.16889571341039453
Confusion Matrix
[[135 365]
 [ 19 481]]
Hallucination False Negative Risk: 0.038

              precision    recall  f1-score   support

           0       0.88      0.27      0.41       500
           1       0.57      0.96      0.71       500

    accuracy                           0.62      1000
   macro avg       0.72      0.62      0.56      1000
weighted avg       0.72      0.62      0.56      1000



## 17. 최종 output 생성

개별 답변 단위의 환각 가능성과 미탐 위험 산출

- "Undetected_Hallucination_Risk": factual로 통과된 답변 안에 숨어 있는 환각 위험 반영

In [23]:
result_df = test_df.copy()

result_df["Hallucination_Probability"] = test_prob
result_df["Predicted_Label"] = test_pred
result_df["Prediction"] = np.where(
    result_df["Predicted_Label"] == 1,
    "Hallucination",
    "Factual"
)

# 모델 전체의 false negative risk
result_df["Hallucination_False_Negative_Risk"] = hallucination_false_negative_risk

# 정상으로 통과된 답변 안에 숨어 있는 미탐 위험
result_df["Undetected_Hallucination_Risk"] = np.where(
    result_df["Predicted_Label"] == 0,
    hallucination_false_negative_risk * (1 - result_df["Hallucination_Probability"]),
    0
)

result_df[
    [
        "Question",
        "Knowledge",
        "Ground Truth",
        "Answer",
        "label",
        "Hallucination_Probability",
        "Predicted_Label",
        "Prediction",
        "Hallucination_False_Negative_Risk",
        "Undetected_Hallucination_Risk"
    ]
].head()


,Question,Knowledge,Ground Truth,Answer,label,Hallucination_Probability,Predicted_Label,Prediction,Hallucination_False_Negative_Risk,Undetected_Hallucination_Risk
1704,Does the enteral feeding advancement affect sh...,[Controversy exists regarding the optimal ente...,Center differences in enteral feeding advancem...,The rapid advancement of enteral feeding in ve...,1,0.841071,1,Hallucination,0.038,0.000000
251,Is gastric electrical stimulation superior to ...,[Severe upper gastrointestinal (GI) motor diso...,We conclude that GES is more effective in impr...,We conclude that GES is more effective in impr...,0,0.121273,0,Factual,0.038,0.033392
1962,Do liquid-based preparations of urinary cytolo...,[The cytomorphology of liquid-based preparatio...,Liquid-based preparations performed significan...,Liquid-based preparations have shown to perfor...,1,0.247435,1,Hallucination,0.038,0.000000
768,Are the elderly with oropharyngeal carcinoma u...,[To determine if elderly patients with orophar...,Proportionally fewer elderly patients with OPS...,Proportionally fewer elderly patients with OPS...,0,0.150572,1,Hallucination,0.038,0.000000
1127,The influence of atmospheric pressure on aorti...,[The rate of aortic aneurysm rupture correlate...,The results of this study do not support the h...,Changes in atmospheric pressure significantly ...,1,0.653501,1,Hallucination,0.038,0.000000


## 18. P1 AI Design Risk Score 연결용 penalty

In [24]:
alpha = 0.6
beta = 0.3
gamma = 0.1

result_df["AI_Decision_Reliability_Penalty"] = (
    alpha * result_df["Hallucination_Probability"]
    + beta * result_df["Hallucination_False_Negative_Risk"]
    + gamma * result_df["Undetected_Hallucination_Risk"]
)

result_df[
    [
        "Hallucination_Probability",
        "Hallucination_False_Negative_Risk",
        "Undetected_Hallucination_Risk",
        "AI_Decision_Reliability_Penalty"
    ]
].head()


,Hallucination_Probability,Hallucination_False_Negative_Risk,Undetected_Hallucination_Risk,AI_Decision_Reliability_Penalty
1704,0.841071,0.038,0.000000,0.516042
251,0.121273,0.038,0.033392,0.087503
1962,0.247435,0.038,0.000000,0.159861
768,0.150572,0.038,0.000000,0.101743
1127,0.653501,0.038,0.000000,0.403500


## 19. Feature importance 확인

In [25]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(30)


,feature,importance
1,sbert_cos_QA,0.030429
407,answer_len,0.017517
390,top3_KA_mean_sim,0.014119
397,num_mismatch_count_KA,0.013236
398,num_mismatch_ratio_KA,0.011083
4,diff_KA_l2,0.011076
408,answer_knowledge_len_ratio,0.009680
389,top1_KA_sim,0.008710
391,top3_KA_max_sim,0.008422
2,diff_KA_mean,0.007081


## 20. 결과 저장

In [27]:
result_df.to_csv("medhal_hallucination_risk_output_fast_final.csv", index=False)
threshold_table.to_csv("threshold_optimization_table_fast_final.csv", index=False)
feature_importance.to_csv("feature_importance_fast_final.csv", index=False)

from google.colab import files

files.download("medhal_hallucination_risk_output_fast_final.csv")
files.download("threshold_optimization_table_fast_final.csv")
files.download("feature_importance_fast_final.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 21. 최종 플로우 요약

```text
MedHal
→ GT는 label/평가에만 사용
→ Q/K/A 기반 SBERT 관계 피처
→ Knowledge top-k sentence similarity
→ symbolic mismatch feature
   - 숫자 불일치
   - 단위 불일치
   - 방향성 불일치
   - 부정 표현 불일치
→ XGBoost
→ probability calibration
→ FN 중심 threshold
→ Hallucination Probability & Hallucination FN Risk
→ P1 Risk Score
```
